# Disky / Spiral Lens — Single-Sersic vs. Two-Component Light

## Learning to Autolens / Examples / disky_spiral_lens

---

**Problem.** Real lens galaxies often have **two light components at different position angles** — a high-Sersic-index bulge and a lower-n disk that's more flattened and rotated. A single-Sersic light profile *cannot* capture this morphology; it's mathematically a single ellipse in surface brightness.

This example shows what that looks like in practice:
- Simulated lens has bulge (PA≈0°, *n*=4, *R_e*=0.45″) + disk (PA≈35°, *n*=1, *R_e*=1.0″).
- Mass is Isothermal + shear, aligned with the bulge PA.
- Source is a compact Sersic at $z=1.6$.

**Pedagogical payoff.** Fit it two ways and compare:
1. **Single Sersic**: leaves coherent residuals at the lens centre where the bulge and disk don't both fit.
2. **Bulge + disk** (two Sersics, independent ell_comps): subtracts the lens light cleanly.

The log-evidence difference quantifies the value of the 7 extra disk parameters. This is the simpler cousin of the Module 09 MGE technique — MGE handles arbitrary morphology with ~10 components; here we just demonstrate the *principle* with 2.

**Prerequisites.** Mod 03 (free fit), Mod 04 (SLaM), Mod 09 (MGE / linear light profiles).


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, IFrame, Markdown, display

import autofit as af
import autolens as al
import autolens.plot as aplt

%matplotlib inline
print(f"PyAutoLens version: {al.__version__}")

In [ ]:
RESULTS_ROOT = Path("results")

def show_result(stage_name):
    stage_dir = RESULTS_ROOT / stage_name
    if not stage_dir.exists():
        print(f"(no results for stage {stage_name!r})")
        return
    sp = stage_dir / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        display(Markdown(
            f"### `{stage_name}`\n"
            f"- log evidence: **{s.get('log_evidence'):.2f}**\n"
            f"- χ²/N = **{s.get('chi_squared_per_pixel'):.3f}**\n"
            f"- max |normalized residual| = **{s.get('max_abs_normalized_residual'):.2f} σ**"
        ))
    fp = stage_dir / "fit_subplot.png"
    if fp.exists():
        display(Image(filename=str(fp)))

---
## 1. The data

A 120×120 HST-like cutout (pixel_scales=0.05″) of the lens + single-source system. Running `mocks/generate_mock.py` regenerates it.

In [ ]:
dataset = al.Imaging.from_fits(
    data_path      = Path("mocks") / "mock_image.fits",
    noise_map_path = Path("mocks") / "mock_noise.fits",
    psf_path       = Path("mocks") / "mock_psf.fits",
    pixel_scales   = 0.05,
)
mask = al.Mask2D.circular(
    shape_native = dataset.shape_native,
    pixel_scales = dataset.pixel_scales,
    radius       = 2.8,
)
dataset = dataset.apply_mask(mask=mask)
aplt.subplot_imaging_dataset(dataset=dataset)

---
## 2. The two model variants

Full details in `Modules/10_Cluster_Computing/scripts/fit_example_disky_spiral_lens.py`.

**Variant 1 — single Sersic (17 free parameters):**

| Component | Profile | Notes |
|---|---|---|
| Lens light | one `al.lp.Sersic` | centre tied, wide ell_comps, n ∈ [0.8, 5] |
| Lens mass | `al.mp.Isothermal` | `einstein_radius ~ UniformPrior(0.5, 3)` |
| Shear | `al.mp.ExternalShear` | Gaussian(0, 0.1) |
| Source | `al.lp.SersicCore` | centre Gaussian(0, 0.3), compact R_e |

**Variant 2 — bulge + disk (24 free parameters):**

| Component | Profile | Notes |
|---|---|---|
| Lens bulge | `al.lp.Sersic` | n ∈ [2, 6], R_e ∈ [0.1, 1.5] |
| Lens disk | `al.lp.Sersic` | n ∈ [0.5, 2], R_e ∈ [0.3, 3], **INDEPENDENT ell_comps** |
| Lens mass + shear + source | same as Variant 1 | |

The bulge and disk share a centre (tied) but have **independent ell_comps**, which is what lets them represent different position angles.

---
## 3. Results (loaded from Cannon)

Submitted via:

```bash
sbatch --export=ALL,EXAMPLE=disky_spiral_lens,FIT_EXTRA_ARGS=--part=all \
    Modules/10_Cluster_Computing/scripts/submit_cannon.slurm
```

`--part=all` runs both variants sequentially into `results/single_sersic_fit/` and `results/bulge_disk_fit/`.

In [ ]:
show_result("single_sersic_fit")

In [ ]:
show_result("bulge_disk_fit")

---
## 4. Comparison

When both fits are in, fill in the table below and compute the Bayes factor.

| | Single Sersic | Bulge + Disk |
|---|---|---|
| log_Z | ___ | ___ |
| χ²/N | ___ | ___ |
| max\|res\| | ___σ | ___σ |
| n_params | 17 | 24 |
| Lens Light Subtracted: central residual? | ___ | ___ |

**Bayes factor** = exp(log_Z_bulge_disk − log_Z_single).

If the factor is large (≫ 1, say e⁵⁰⁺), the extra disk component is strongly favoured — the 7 extra parameters buy substantial likelihood. If the factor is near 1 (or favours Variant 1), the extra complexity isn't needed and the data don't resolve a disk.

### What you'd expect for this mock

- Variant 1 `chi²/N ≫ 1` with coherent lens-centre residual (the bulge-only model systematically over- or under-predicts the disk flux at different azimuths).
- Variant 2 `chi²/N ≈ 1` with salt-and-pepper residual.
- Bayes factor strongly favours Variant 2 — the data have enough S/N to distinguish the two-PA structure.

---

## 5. Exercises

1. **How disparate do the two PAs have to be before Variant 1 fails?** Regenerate the mock (`generate_mock.py`) with a smaller PA difference — say 10° between bulge and disk, instead of 35°. Refit both variants. At what PA difference does the Bayes factor flip to prefer Variant 1? (Below the threshold, a single elliptical Sersic is a sufficient approximation.)

2. **Mass–light PA misalignment sanity check.** The mock has mass_PA = bulge_PA, not mass_PA = total_light_PA. If you had *only* Variant 1's single-Sersic lens light (which averages the two PAs), would you incorrectly tie mass_PA to that averaged light_PA? How much would that bias the recovered Einstein radius?

3. **Compare to MGE.** Module 09 shows MGE (Multi-Gaussian Expansion) handling arbitrary lens morphology with ~10 Gaussian components. Fit the same mock with an MGE-light Lens and compare log_Z to the 2-Sersic Variant 2 here. MGE should either match or beat.

---

## References

- Cappellari (2002), MNRAS 333, 400 — MGE parameterisation.
- Nightingale+18 §3 — parametric light profiles in PyAutoLens.
- `Modules/09_MGE_Linear_Light_Profiles/` — MGE pipeline in this repo.

---

*Learning to Autolens — Examples / disky_spiral_lens / 01*  
*Rodrigo Córdova Rosado, Harvard CfA*